In [1]:
""" 0. set-up part:  import necessary libraries and set up environment """

import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag, word_tokenize
from collections import Counter, defaultdict
import numpy as np
import math
import copy
import itertools
import matplotlib.pyplot as plt
import matplotlib as mpl

import joblib
from joblib import Parallel, delayed
from threading import Thread

import os
import pickle
import time

import operator
from functools import reduce
import json
import cProfile

import gensim
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora import Dictionary

import tomotopy as tp

# download nltk data once time
# nltk.download('punkt')
# nltk.download('stopwords')
# nltk.download('wordnet')
# nltk.download('averaged_perceptron_tagger')
# nltk.download('omw-1.4')
# nltk.download('punkt_tab')
# nltk.download('averaged_perceptron_tagger_eng')

#  chinese character support in matplotlib
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS' 'SimHei' 'DejaVu Sans']  
plt.rcParams['axes.unicode_minus'] = False

In [2]:
""" 1.1 Data Preprocessing: load data, clean text, lemmatization, remove low-frequency words"""

# Map POS tags to WordNet format， Penn Treebank annotation: fine-grained (45 tags), WordNet annotation: coarse-grained (4 tags: a, v, n, r)
def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):
        return 'a'  # 形容词
    elif treebank_tag.startswith('V'):
        return 'v'  # 动词
    elif treebank_tag.startswith('N'):
        return 'n'  # 名词
    elif treebank_tag.startswith('R'):
        return 'r'  # 副词
    else:
        return 'n'  # 默认名词

# Text cleaning and lemmatization preprocessing function
def clean_and_lemmatize(text):
    if pd.isnull(text):
        return []
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)  # Remove non-alphabetic characters using regex
    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w not in stop_words]
    pos_tags = pos_tag(tokens)
    lemmatized = [lemmatizer.lemmatize(w, get_wordnet_pos(pos)) for w, pos in pos_tags]
    return lemmatized  

#-----------------Load data----------------
data = pd.read_excel('./data/raw/papers_CM.xlsx', usecols=['PaperID', 'Abstract', 'Keywords', 'Year'])

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

# clean and lemmatize the abstracts
data['Lemmatized_Tokens'] = data['Abstract'].apply(clean_and_lemmatize)

# count word frequencies
all_tokens = [word for tokens in data['Lemmatized_Tokens'] for word in tokens]
word_counts = Counter(all_tokens)

# set a minimum frequency threshold for valid words
min_freq = 10
valid_words = set([word for word, freq in word_counts.items() if freq >= min_freq])

# remove rare words based on frequency threshold
def remove_rare_words(tokens):
    return [word for word in tokens if word in valid_words]

data['Filtered_Tokens'] = data['Lemmatized_Tokens'].apply(remove_rare_words)

# join tokens back into cleaned abstracts
data['Cleaned_Abstract'] = data['Filtered_Tokens'].apply(lambda x: " ".join(x))

# create a cleaned DataFrame with relevant columns
cleaned_data = data[['PaperID', 'Year', 'Cleaned_Abstract']]
cleaned_data = cleaned_data[~(cleaned_data['PaperID'] == 57188)] # this paper has no abstract
cleaned_data = cleaned_data.reset_index(drop=True) 
cleaned_data.insert(0, 'Document_ID', range(len(cleaned_data))) 
abstract_list = cleaned_data['Cleaned_Abstract'].apply(lambda x: x.split()).tolist()

corpus = {doc_id: abstract_list for doc_id, abstract_list in enumerate(abstract_list)}
# cleaned_data.to_csv('./data/processed/cleaned_data.xlsx', index=False, encoding='utf-8-sig')

In [3]:
# ===== Enhanced Coherence Calculation Function (Supports Multiple Metrics Including NPMI) =====
def calculate_multiple_coherence_metrics(mdl, corpus_docs, metrics=['c_v', 'c_npmi'], fast_mode=True, top_n=10):
    """
    Enhanced coherence calculation function - supports multiple coherence metrics including NPMI
    
    Supported coherence metrics:
    - c_v: Vector space-based coherence (default)
    - c_npmi: Normalized Pointwise Mutual Information (NPMI)
    
    Supported model types:
    - Single-layer models: LDA, CTM
    - Hierarchical models: hLDA, PAM, hPAM
    - Non-parametric models: HDP
    
    Returns: dict containing various coherence metrics
    """
    try:
        docs = list(corpus_docs)
        dictionary = Dictionary(docs)
        
        # Check if it is a hierarchical model
        model_type_str = str(type(mdl))
        is_hierarchical = hasattr(mdl, 'depth') or 'HLDA' in model_type_str or 'PAM' in model_type_str
        
        if is_hierarchical:
            # Hierarchical model: calculate coherence by level, weighted average
            metrics_results = {metric: [] for metric in metrics}
            layer_weights = []
            
            try:
                for level in range(getattr(mdl, 'depth', 3)):
                    level_topics = []
                    level_doc_count = 0
                    
                    # Iterate through all nodes of this level
                    for k in range(getattr(mdl, 'k', 100)):
                        try:
                            topic_words = mdl.get_topic_words(k, top_n=top_n)
                            if topic_words:
                                words = [word for word, prob in topic_words]
                                level_topics.append(words)
                                level_doc_count += 1
                        except:
                            continue
                    
                    if level_topics:
                        # Calculate coherence for each metric
                        for metric in metrics:
                            try:
                                cm = CoherenceModel(
                                    topics=level_topics,
                                    texts=docs,
                                    dictionary=dictionary,
                                    coherence=metric,
                                    processes=1
                                )
                                score = cm.get_coherence()
                                if score and not math.isnan(score):
                                    metrics_results[metric].append(score)
                                else:
                                    metrics_results[metric].append(0.1)
                            except Exception as e:
                                print(f"Hierarchical model {metric} calculation warning: {e}")
                                metrics_results[metric].append(0.1)
                        
                        layer_weights.append(level_doc_count)
            except:
                pass
            
            # Calculate weighted average
            final_results = {}
            for metric in metrics:
                if metrics_results[metric] and layer_weights:
                    total_weight = sum(layer_weights)
                    if total_weight > 0:
                        weighted_avg = sum(score * w for score, w in zip(metrics_results[metric], layer_weights)) / total_weight
                        final_results[metric] = weighted_avg
                    else:
                        final_results[metric] = 0.1
                else:
                    final_results[metric] = 0.1
            
            return final_results
        
        # Single-layer model: calculate coherence for all topics
        num_topics = getattr(mdl, 'k', None) or getattr(mdl, 'num_topics', None) or 100
        topics = []
        
        for k in range(num_topics):
            try:
                topic_words = mdl.get_topic_words(k, top_n=top_n)
                if topic_words:
                    words = [word for word, prob in topic_words]
                    topics.append(words)
            except:
                continue
        
        if not topics:
            return {metric: 0.1 for metric in metrics}
        
        # Calculate coherence for each metric
        final_results = {}
        for metric in metrics:
            try:
                cm = CoherenceModel(
                    topics=topics,
                    texts=docs,
                    dictionary=dictionary,
                    coherence=metric,
                    processes=1
                )
                score = cm.get_coherence()
                final_results[metric] = score if score and not math.isnan(score) else 0.1
            except Exception as e:
                print(f"Single-layer model {metric} calculation warning: {e}")
                final_results[metric] = 0.1
        
        return final_results
        
    except Exception as e:
        print(f"Coherence calculation error: {e}")
        return {metric: 0.1 for metric in metrics}

In [4]:
def calculate_renyi_entropy_unweighted(model, alpha=2):
    """
    Calculate the Renyi entropy for all topics (unweighted version, direct average)
    - model: topic model object (e.g., tomotopy LDA/CTM/PAM/hLDA)
    - alpha: order of Renyi entropy (commonly 2)
    Returns: the average of Renyi entropies for all topics
    """
    # ...existing code...
    import numpy as np
    entropies = []
    num_topics = getattr(model, 'k', None) or getattr(model, 'num_topics', None) or 0
    for k in range(num_topics):
        try:
            topic_probs = np.array([prob for word, prob in model.get_topic_words(k, top_n=-1)])
            topic_probs = topic_probs / topic_probs.sum()
            if len(topic_probs) > 0:
                renyi = (1/(1-alpha)) * np.log(np.sum(topic_probs**alpha))
                entropies.append(renyi)
            else:
                entropies.append(0)
        except:
            entropies.append(0)
    if entropies:
        return float(np.mean(entropies))
    else:
        return 0.0

In [5]:
# ===== 加权Renyi熵计算函数 (Weighted Renyi Entropy) =====
def get_topic_doc_counts(model, threshold=0.01):
    """
    获取每个主题覆盖的文档数（即每个主题出现在了多少个文档中）
    """
    num_topics = getattr(model, 'k', None) or getattr(model, 'num_topics', None) or 0
    topic_doc_counts = [0] * num_topics
    for doc in model.docs:
        try:
            topic_dist = doc.get_topic_dist()
        except:
            continue
        for k, prob in enumerate(topic_dist):
            if prob > threshold:
                topic_doc_counts[k] += 1
    return topic_doc_counts

def calculate_weighted_renyi_entropy(model, alpha=2):
    """
    按主题覆盖的文档数加权的Renyi熵
    """
    entropies = []
    num_topics = getattr(model, 'k', None) or getattr(model, 'num_topics', None) or 0
    for k in range(num_topics):
        try:
            topic_probs = np.array([prob for word, prob in model.get_topic_words(k, top_n=-1)])
            if topic_probs.sum() > 0:
                topic_probs = topic_probs / topic_probs.sum()
                renyi = (1/(1-alpha)) * np.log(np.sum(topic_probs**alpha))
                entropies.append(renyi)
            else:
                entropies.append(0)
        except:
            entropies.append(0)
    
    # 按主题的文档覆盖数进行加权
    topic_doc_counts = get_topic_doc_counts(model)
    total_docs_covered = sum(topic_doc_counts)
    
    if total_docs_covered > 0 and len(entropies) == len(topic_doc_counts):
        return np.average(entropies, weights=topic_doc_counts)
    elif entropies:
        return np.mean(entropies) # 如果加权失败，则返回普通平均值
    else:
        return 0.0

In [6]:
from scipy.spatial.distance import jensenshannon
import numpy as np

def calculate_topic_diversity_jsd(model, top_n=25):
    """
    计算所有主题对之间的平均JSD（Jensen-Shannon Divergence），以衡量主题多样性。
    JSD值域为[0, 1]，值越高代表主题间差异越大，多样性越好。

    - model: 训练好的tomotopy模型。
    - top_n: 用于计算JSD的每个主题的top N个词。
    """
    try:
        num_topics = getattr(model, 'k', None) or getattr(model, 'num_topics', None) or 0
        if num_topics < 2:
            return 0.0

        # 1. 收集所有主题的top N词和概率，并建立一个共享词汇表
        topic_word_probs = []
        vocab = set()
        for k in range(num_topics):
            try:
                words = model.get_topic_words(k, top_n=top_n)
                if not words: continue
                topic_word_probs.append(dict(words))
                vocab.update([word for word, prob in words])
            except:
                continue
        
        if len(topic_word_probs) < 2:
            return 0.0

        vocab_list = sorted(list(vocab))
        vocab_map = {word: i for i, word in enumerate(vocab_list)}
        
        # 2. 将每个主题的词分布转换为对齐的概率向量
        aligned_probs = []
        for topic_dict in topic_word_probs:
            prob_vector = np.zeros(len(vocab_list))
            for word, prob in topic_dict.items():
                if word in vocab_map:
                    prob_vector[vocab_map[word]] = prob
            
            # 归一化，使其成为有效的概率分布
            if prob_vector.sum() > 1e-9:
                prob_vector /= prob_vector.sum()
            else:
                continue # 跳过无效的空主题
            aligned_probs.append(prob_vector)

        if len(aligned_probs) < 2:
            return 0.0

        # 3. 计算所有主题对之间的JSD
        jsd_values = []
        for i in range(len(aligned_probs)):
            for j in range(i + 1, len(aligned_probs)):
                p = aligned_probs[i]
                q = aligned_probs[j]
                jsd = jensenshannon(p, q, base=2)
                if not np.isnan(jsd):
                    jsd_values.append(jsd**2) # JSD距离通常使用JSD值的平方

        if not jsd_values:
            return 0.0
        
        return float(np.mean(jsd_values))

    except Exception as e:
        # print(f"Error calculating JSD: {e}")
        return 0.0

In [7]:
# ===== General Topic Analysis Function =====
def analyze_model_topics(model, model_name="Model", top_words=5, min_prob=0.01, max_display=10):
    """
    General topic analysis function - applicable to all topic models
    
    Functionality:
    - Extracts active topics
    - Displays topic words and weights
    - Calculates topic activity rate
    """
    print(f"\n🔍 {model_name} Topic Analysis (showing top {top_words} words):")
    print("=" * 80)
    
    active_topics = 0
    topic_info = []
    
    num_topics = getattr(model, 'k', None) or getattr(model, 'num_topics', None) or 0
    
    for k in range(num_topics):
        try:
            topic_words = model.get_topic_words(k, top_n=top_words)
            if topic_words and topic_words[0][1] > min_prob:
                active_topics += 1
                words_str = ", ".join([f"{word}({prob:.3f})" for word, prob in topic_words[:5]])
                topic_info.append((k, topic_words[0][1], words_str))
                
                if active_topics <= max_display:
                    print(f"Topic {k:3d} (weight:{topic_words[0][1]:.3f}): {words_str}")
        except:
            continue
    
    if active_topics > max_display:
        print(f"... (and {active_topics - max_display} more active topics)")
    
    print(f"\n📊 {model_name} Topic Statistics:")
    print(f"   - Active topics: {active_topics}/{num_topics}")
    print(f"   - Topic activity rate: {active_topics/num_topics*100:.1f}%")
    
    return topic_info

In [8]:
def calculate_r_hat(chains_history):
    """
    根据多个MCMC链的后半部分历史记录计算R-hat（Gelman-Rubin诊断）值。
    
    参数:
    - chains_history (np.ndarray): 一个2D numpy数组，形状为 (M, N)，
      其中 M 是链的数量（运行次数），N 是每个链记录的迭代次数。
      数组中的值是对数似然（log-likelihood）。

    返回:
    - float: R-hat值。如果无法计算，则返回 np.nan。
    """
    # 仅使用后半部分的样本进行计算，这是标准做法
    num_iterations = chains_history.shape[1]
    start_index = num_iterations // 2
    
    if chains_history.shape[0] < 2 or (num_iterations - start_index) < 2:
        return np.nan

    chains_history = chains_history[:, start_index:]
    num_chains, num_used_iterations = chains_history.shape
    
    # 1. 计算每个链的均值
    chain_means = np.mean(chains_history, axis=1)
    
    # 2. 计算每个链的方差
    chain_variances = np.var(chains_history, axis=1, ddof=1)
    
    # 3. 计算链内方差的均值 (W)
    W = np.mean(chain_variances)
    
    # 4. 计算链间方差 (B)
    overall_mean = np.mean(chain_means)
    B = num_used_iterations / (num_chains - 1) * np.sum((chain_means - overall_mean)**2)
    
    # 5. 估计目标分布的方差 (Var_hat)
    var_hat = (1 - 1 / num_used_iterations) * W + (1 / num_used_iterations) * B
    
    if W == 0:
        return np.nan
        
    # 6. 计算R-hat
    r_hat = np.sqrt(var_hat / W)
    
    return r_hat

In [9]:
def calculate_weighted_coherence(model, corpus_docs, metrics=['c_v', 'c_npmi'], top_n=10, threshold=0.01):
    """
    计算按主题的文档覆盖率加权的 coherence 分数 (NPMI, C_v)。
    
    Args:
        model: 训练好的 tomotopy 模型。
        corpus_docs: 用于计算 coherence 的原始文档列表。
        metrics: 要计算的指标列表。
        top_n: 用于定义主题的 top N 个词。
        threshold: 判断一个主题在文档中是否“显著”的概率阈值。

    Returns:
        一个包含加权和未加权 coherence 分数的字典。
    """
    try:
        docs = list(corpus_docs)
        dictionary = Dictionary(docs)
        num_topics = getattr(model, 'k', 0) or getattr(model, 'num_topics', 0)
        
        if num_topics == 0:
            return {}

        # 1. 提取所有主题的 top words
        topics = []
        for k in range(num_topics):
            topic_words = [word for word, prob in model.get_topic_words(k, top_n=top_n)]
            if topic_words:
                topics.append(topic_words)
        
        if not topics:
            return {}

        # 2. 获取每个主题的权重 (文档覆盖数)
        topic_doc_counts = np.zeros(num_topics)
        for doc in model.docs:
            topic_dist = doc.get_topic_dist()
            for k, prob in enumerate(topic_dist):
                if prob > threshold:
                    topic_doc_counts[k] += 1
        
        total_docs_covered = np.sum(topic_doc_counts)
        
        results = {}
        for metric in metrics:
            cm = CoherenceModel(
                topics=topics,
                texts=docs,
                dictionary=dictionary,
                coherence=metric,
                processes=1
            )
            
            # 获取每个主题的分数
            per_topic_scores = cm.get_coherence_per_topic()
            
            # 计算简单平均值 (未加权)
            unweighted_avg = np.mean(per_topic_scores)
            results[f'unweighted_{metric}'] = unweighted_avg
            
            # 计算加权平均值
            if total_docs_covered > 0:
                weighted_avg = np.average(per_topic_scores, weights=topic_doc_counts)
                results[f'weighted_{metric}'] = weighted_avg
            else:
                results[f'weighted_{metric}'] = unweighted_avg # 如果没有权重，则退回到简单平均

        return results

    except Exception as e:
        print(f"计算加权Coherence时出错: {e}")
        return {}

# --- 如何使用 ---
# 假设 best_lda_model 是你训练好的模型
# weighted_scores = calculate_weighted_coherence(best_lda_model, abstract_list)
# print(weighted_scores)
# 输出可能像这样:
# {'unweighted_c_v': 0.45, 'weighted_c_v': 0.48, 'unweighted_c_npmi': 0.05, 'weighted_c_npmi': 0.06}

In [10]:
def tune_hdp_model_grid_search(
    docs, 
    gamma_range, 
    alpha_range, 
    eta_range, 
    num_runs=3, 
    seed=42, 
    max_iters=1000,
    burn_in=100,
    convergence_patience=5,
    convergence_tolerance=1e-3
):
    """
    对HDP模型执行全面的网格搜索。

    - 遍历所有 gamma, alpha, eta 的组合。
    - 包含一个独立的 burn-in 阶段。
    - 使用对数似然（LL）的变化判断收敛。
    - 为每个组合运行多次以获得稳健的结果。
    - 收集包括 LL/word, 加权/非加权 NPMI/Cv, PPL, Renyi熵, JSD多样性在内的详细指标。
    """
    print("="*80)
    print("🚀 开始 HDP 模型全面网格搜索")
    print("="*80)

    all_run_results = []
    best_single_run_npmi = -1
    best_model = None
    best_params = {}
    
    param_combinations = list(itertools.product(gamma_range, alpha_range, eta_range))
    total_combinations = len(param_combinations)

    for i, (gamma, alpha, eta) in enumerate(param_combinations, 1):
        print(f"\n--- 组合 {i}/{total_combinations}: 测试 Gamma={gamma}, Alpha={alpha}, Eta={eta}，运行 {num_runs} 次 ---")
        
        for run_idx in range(num_runs):
            start_time = time.time()
            current_seed = seed + run_idx
            print(f"  - 运行 {run_idx + 1}/{num_runs} (seed={current_seed})...")
            
            try:
                model = tp.HDPModel(gamma=gamma, alpha=alpha, eta=eta, seed=current_seed)
                for doc in docs:
                    model.add_doc(doc)
                
                # Burn-in 阶段
                if burn_in > 0:
                    print(f"    => 执行 {burn_in} 次 Burn-in 迭代...")
                    model.train(burn_in)
                
                # 动态训练与收敛判断
                ll_history = []
                converged = False
                final_iter = max_iters
                for step in range(0, max_iters, 10):
                    model.train(10, 1) # 每次训练10次迭代，采集1次样本
                    ll_history.append(model.ll_per_word)
                    if len(ll_history) > convergence_patience:
                        recent_ll = ll_history[-convergence_patience:]
                        if (np.max(recent_ll) - np.min(recent_ll)) < convergence_tolerance:
                            final_iter = step + 10
                            print(f"    => 在第 {final_iter} 次迭代时收敛 (总迭代: {burn_in + final_iter})")
                            converged = True
                            break
                
                if not converged:
                    final_iter = max_iters
                    print(f"    => 达到最大迭代次数 {max_iters}")

                # 评估指标
                ll_per_word = model.ll_per_word
                perplexity = math.exp(-ll_per_word)
                coherence_scores = calculate_weighted_coherence(model, docs, metrics=['c_v', 'c_npmi'])
                renyi_entropy = calculate_weighted_renyi_entropy(model)
                jsd_diversity = calculate_topic_diversity_jsd(model)
                
                w_npmi = coherence_scores.get('weighted_c_npmi', 0)
                
                print(f"    => 结果: Live_Topics={model.live_k}, LL/word={ll_per_word:.4f}, Weighted_NPMI={w_npmi:.4f}, PPL={perplexity:.2f}, Renyi={renyi_entropy:.4f}, JSD={jsd_diversity:.4f}, Converged={'Yes' if converged else 'No'}")

                # 记录所有指标
                run_result = {
                    'Gamma': gamma, 'Alpha': alpha, 'Eta': eta,
                    'Run_Index': run_idx + 1,
                    'Live_Topics': model.live_k,
                    'LL_per_word': ll_per_word,
                    'Weighted_NPMI': coherence_scores.get('weighted_c_npmi', 0),
                    'Weighted_Cv': coherence_scores.get('weighted_c_v', 0),
                    'Unweighted_NPMI': coherence_scores.get('unweighted_c_npmi', 0),
                    'Unweighted_Cv': coherence_scores.get('unweighted_c_v', 0),
                    'Perplexity': perplexity,
                    'Weighted_Renyi_Entropy': renyi_entropy,
                    'JSD_Diversity': jsd_diversity,
                    'Iterations': final_iter + burn_in,
                    'Converged': "Yes" if converged else "No",
                    'Time_sec': time.time() - start_time
                }
                all_run_results.append(run_result)
                
                # 追踪在所有运行中NPMI最高的模型
                if w_npmi > best_single_run_npmi:
                    best_single_run_npmi = w_npmi
                    best_params = {'Gamma': gamma, 'Alpha': alpha, 'Eta': eta}
                    best_model = model

            except Exception as e:
                print(f"    ❌ 运行 {run_idx + 1} 失败: {e}")

    results_df = pd.DataFrame(all_run_results)
    print("\n\n✅ HDP 全面网格搜索完成!")
    
    if best_params:
        print(f"\n🏆 所有运行中 Weighted_NPMI 最高的一次参数: Gamma={best_params['Gamma']}, Alpha={best_params['Alpha']}, Eta={best_params['Eta']} (NPMI: {best_single_run_npmi:.4f})")
    else:
        print("\n❌ 未能成功完成任何运行。")

    return results_df, best_model, best_params

In [11]:
# ===== HDP 全面网格搜索实例 =====
print("\n🎯 开始为 HDP 模型进行全面网格搜索...")
print("=" * 60)

# 1. 定义需要测试的超参数范围
gamma_range_to_test = [0.01, 0.05, 0.1, 0.5, 1, 2, 5, 10, 20]
alpha_range_to_test = [1]
eta_range_to_test = [0.1]


# 2. 确保您的预处理文档已准备就绪
hdp_docs = abstract_list

# 3. 运行网格搜索函数
#    - num_runs=3: 每个参数组合运行3次以获得更稳定的结果
#    - max_iters=1000: 训练迭代次数
hdp_results_df, best_hdp_model, best_hdp_params = tune_hdp_model_grid_search(
    docs=hdp_docs,
    gamma_range=gamma_range_to_test,
    alpha_range=alpha_range_to_test,
    eta_range=eta_range_to_test,
    num_runs=5,
    max_iters=2000,
    burn_in = 1000
)

# 4. 打印找到的最佳模型信息（基于单次最高NPMI） 
print("\n\n🎉 HDP 网格搜索完成！")
if best_hdp_params:
    print(f"🏆 找到的最佳超参数 (基于全局单次运行最高Weighted_NPMI): "
          f"Gamma={best_hdp_params.get('Gamma')}, "
          f"Alpha={best_hdp_params.get('Alpha')}, "
          f"Eta={best_hdp_params.get('Eta')}")
    
    # 使用之前定义的 analyze_model_topics 函数分析最佳模型
    print("\n🔍 对最佳HDP模型进行主题分析:")
    analyze_model_topics(best_hdp_model, model_name="Best HDP", top_words=10)
else:
    print("❌ 未能找到最佳模型。")

# 5. 显示并保存详细的、每一次运行的调优结果表格
if hdp_results_df is not None and not hdp_results_df.empty:
    print("\n--- HDP 最终全部运行调优结果 ---")
    # 调整列顺序以获得更好的可读性
    cols = [
        'Gamma', 'Alpha', 'Eta', 'Run_Index', 'Live_Topics', 
        'Weighted_NPMI', 'Weighted_Cv', 'Perplexity', 
        'Weighted_Renyi_Entropy', 'JSD_Diversity', 'LL_per_word', 
        'Converged', 'Iterations', 'Time_sec',
        'Unweighted_NPMI', 'Unweighted_Cv'
    ]
    display_df = hdp_results_df.reindex(columns=[c for c in cols if c in hdp_results_df.columns])
    print(display_df.to_string())
    
    # 保存结果
    output_path = './data/model_result/hdp_grid_search_results.csv'
    hdp_results_df.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"\n✅ 结果已保存至: {output_path}")


🎯 开始为 HDP 模型进行全面网格搜索...
🚀 开始 HDP 模型全面网格搜索

--- 组合 1/9: 测试 Gamma=0.01, Alpha=1, Eta=0.1，运行 5 次 ---
  - 运行 1/5 (seed=42)...
    => 执行 1000 次 Burn-in 迭代...
    => 在第 60 次迭代时收敛 (总迭代: 1060)


/var/folders/v5/6mdkg5713kxgwg5xs24g8rvr0000gn/T/ipykernel_61540/2801231160.py:50: RuntimeWarning: The training result may differ even with fixed seed if `workers` != 1.
  model.train(burn_in)


    => 结果: Live_Topics=5, LL/word=-6.6203, Weighted_NPMI=0.0136, PPL=750.18, Renyi=5.6663, JSD=0.7093, Converged=Yes
  - 运行 2/5 (seed=43)...
    => 执行 1000 次 Burn-in 迭代...
    => 在第 60 次迭代时收敛 (总迭代: 1060)
    => 结果: Live_Topics=7, LL/word=-6.6309, Weighted_NPMI=0.0223, PPL=758.19, Renyi=5.6850, JSD=0.7923, Converged=Yes
  - 运行 3/5 (seed=44)...
    => 执行 1000 次 Burn-in 迭代...
    => 在第 60 次迭代时收敛 (总迭代: 1060)
    => 结果: Live_Topics=5, LL/word=-6.6363, Weighted_NPMI=0.0219, PPL=762.28, Renyi=5.6860, JSD=0.7933, Converged=Yes
  - 运行 4/5 (seed=45)...
    => 执行 1000 次 Burn-in 迭代...
    => 在第 140 次迭代时收敛 (总迭代: 1140)
    => 结果: Live_Topics=9, LL/word=-6.6419, Weighted_NPMI=0.0119, PPL=766.56, Renyi=5.6680, JSD=0.9346, Converged=Yes
  - 运行 5/5 (seed=46)...
    => 执行 1000 次 Burn-in 迭代...
    => 在第 80 次迭代时收敛 (总迭代: 1080)
    => 结果: Live_Topics=5, LL/word=-6.6458, Weighted_NPMI=0.0061, PPL=769.55, Renyi=5.6692, JSD=0.6711, Converged=Yes

--- 组合 2/9: 测试 Gamma=0.05, Alpha=1, Eta=0.1，运行 5 次 ---
  - 运行 1/5